In [ ]:
import numpy as np

export_path = "/nfs/home/victor.franco/moment_input.npz"  # ou o caminho completo

data = np.load(export_path, allow_pickle=True)

X = data["X"]                       # (n_janelas, window_size, n_variaveis)
file_name = data["file_name"]       # (n_janelas,)
relative_path = data["relative_path"]
window_idx = data["window_idx"]
signal_columns = data["signal_columns"]

print("X shape:", X.shape)
print("n variáveis:", len(signal_columns))
print("primeiras variáveis:", signal_columns[:5])
print("primeira janela veio de:", relative_path[0], "arquivo:", file_name[0], "window:", window_idx[0])


In [ ]:
import numpy as np
import torch
import torch.cuda.amp as amp
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import OneCycleLR
from tqdm import tqdm

from momentfm import MOMENTPipeline
from momentfm.utils.utils import control_randomness

# =========================================================
# Configurações
# =========================================================
SEED = 13
BATCH_SIZE = 32
MAX_EPOCHS = 3
LR = 1e-4
MAX_NORM = 5.0
TRAIN_RATIO = 0.8
N_CHANNELS = 17
WINDOW = 128
FORECAST_HORIZON = 128

control_randomness(seed=SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# =========================================================
# Checagens de entrada
# Esperado: X com shape (N, 128, 17)
# =========================================================
assert isinstance(X, np.ndarray), "X precisa ser numpy.ndarray"
assert X.ndim == 3, f"Esperado X com 3 dimensões (N, T, C), mas veio {X.shape}"
assert X.shape[1] == WINDOW, f"Esperado janela {WINDOW}, mas veio {X.shape[1]}"
assert X.shape[2] == N_CHANNELS, f"Esperado {N_CHANNELS} canais, mas veio {X.shape[2]}"

X = X.astype(np.float32)

# =========================================================
# Split treino/teste
# =========================================================
n_samples = X.shape[0]
train_end = int(n_samples * TRAIN_RATIO)

X_train_raw = X[:train_end]
X_test_raw = X[train_end:]

# normalização usando apenas treino
mean = X_train_raw.mean(axis=(0, 1), keepdims=True)   # (1, 1, 17)
std = X_train_raw.std(axis=(0, 1), keepdims=True)     # (1, 1, 17)
std = np.where(std < 1e-8, 1.0, std)

X_norm = (X - mean) / std

# =========================================================
# Dataset
# Usa X[i] como passado e X[i+1] como futuro
# =========================================================
class WindowToNextWindowForecastDataset(Dataset):
    def __init__(self, X):
        self.X = torch.tensor(X, dtype=torch.float32)

    def __len__(self):
        return len(self.X) - 1

    def __getitem__(self, idx):
        past = self.X[idx]        # (128, 17)
        future = self.X[idx + 1]  # (128, 17)
        input_mask = torch.ones(past.shape[0], dtype=torch.long)  # (128,)
        return past, future, input_mask

train_dataset = WindowToNextWindowForecastDataset(X_norm[:train_end])
test_dataset = WindowToNextWindowForecastDataset(X_norm[train_end:])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=False,num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False,num_workers=4)

print("train samples:", len(train_dataset))
print("test samples:", len(test_dataset))

# =========================================================
# Modelo
# =========================================================
model = MOMENTPipeline.from_pretrained(
    "AutonLab/MOMENT-1-large",
    model_kwargs={
        "task_name": "forecasting",
        "forecast_horizon": FORECAST_HORIZON,
        "head_dropout": 0.1,
        "n_channels": N_CHANNELS,
        "weight_decay": 0,
        "freeze_encoder": True,
        "freeze_embedder": True,
        "freeze_head": False,
    },
)
model = model.to(device)

criterion = torch.nn.MSELoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

total_steps = len(train_loader) * MAX_EPOCHS
scheduler = OneCycleLR(
    optimizer,
    max_lr=LR,
    total_steps=total_steps,
    pct_start=0.3
)

scaler = amp.GradScaler()

# =========================================================
# Funções auxiliares
# =========================================================
def compute_metrics(y_true, y_pred):
    mse = np.mean((y_true - y_pred) ** 2)
    mae = np.mean(np.abs(y_true - y_pred))
    return mse, mae

def get_forecast_prediction(model, timeseries, input_mask):
    """
    timeseries: tensor (B, 17, 128)
    input_mask: tensor (B, 128)

    Retorna pred com shape compatível com forecast target.
    """
    pred = None

    # Tentativa 1: método específico forecast do pipeline
    if hasattr(model, "forecast") and callable(getattr(model, "forecast")):
        out = model.forecast(x_enc=timeseries, input_mask=input_mask)

        # caso retorne tensor diretamente
        if isinstance(out, torch.Tensor):
            pred = out
        # caso retorne objeto com atributo forecast
        elif hasattr(out, "forecast") and out.forecast is not None:
            pred = out.forecast
        # outras possibilidades defensivas
        elif hasattr(out, "logits") and out.logits is not None:
            pred = out.logits
        elif hasattr(out, "reconstruction") and out.reconstruction is not None:
            pred = out.reconstruction

    # Tentativa 2: forward genérico
    if pred is None:
        out = model(x_enc=timeseries, input_mask=input_mask)

        if hasattr(out, "forecast") and out.forecast is not None:
            pred = out.forecast
        elif hasattr(out, "logits") and out.logits is not None:
            pred = out.logits

    if pred is None:
        raise ValueError(
            "Não foi possível obter previsão do modelo. "
            "O pipeline retornou reconstruction no forward, mas não forecast. "
            "Verifique se o método model.forecast existe e qual assinatura ele espera."
        )

    return pred

# =========================================================
# Diagnóstico rápido antes do treino
# =========================================================
timeseries_dbg, forecast_dbg, input_mask_dbg = next(iter(train_loader))
timeseries_dbg = timeseries_dbg.float().to(device).permute(0, 2, 1)  # (B, 17, 128)
forecast_dbg = forecast_dbg.float().to(device)                       # (B, 128, 17)
input_mask_dbg = input_mask_dbg.to(device)

with torch.no_grad():
    pred_dbg = get_forecast_prediction(model, timeseries_dbg, input_mask_dbg)

print("DEBUG timeseries shape:", tuple(timeseries_dbg.shape))
print("DEBUG target forecast shape:", tuple(forecast_dbg.shape))
print("DEBUG pred shape:", tuple(pred_dbg.shape))

# Ajuste de shape esperado
# alvo original está em (B, 128, 17)
# se predição vier em (B, 17, 128), permutamos o alvo
target_mode = None
if pred_dbg.ndim != 3:
    raise ValueError(f"Predição com ndim inesperado: {pred_dbg.ndim}, shape={pred_dbg.shape}")

if pred_dbg.shape[1] == N_CHANNELS and pred_dbg.shape[2] == FORECAST_HORIZON:
    target_mode = "BCT"  # pred = (B, C, T)
elif pred_dbg.shape[1] == FORECAST_HORIZON and pred_dbg.shape[2] == N_CHANNELS:
    target_mode = "BTC"  # pred = (B, T, C)
else:
    raise ValueError(
        f"Shape de predição inesperado: {pred_dbg.shape}. "
        f"Esperado algo como (B, {N_CHANNELS}, {FORECAST_HORIZON}) "
        f"ou (B, {FORECAST_HORIZON}, {N_CHANNELS})."
    )

print("DEBUG target_mode:", target_mode)

# =========================================================
# Treino e avaliação
# =========================================================
for epoch in range(MAX_EPOCHS):
    model.train()
    train_losses = []

    for timeseries, forecast, input_mask in tqdm(train_loader, desc=f"Train {epoch+1}/{MAX_EPOCHS}"):
        # dataset entrega:
        # timeseries -> (B, 128, 17)
        # forecast   -> (B, 128, 17)
        # input_mask -> (B, 128)

        timeseries = timeseries.float().to(device)
        forecast = forecast.float().to(device)
        input_mask = input_mask.to(device)

        # entrada do MOMENT
        timeseries = timeseries.permute(0, 2, 1)  # (B, 17, 128)

        # ajusta alvo para casar com a saída do modelo
        if target_mode == "BCT":
            target = forecast.permute(0, 2, 1)    # (B, 17, 128)
        else:
            target = forecast                     # (B, 128, 17)

        optimizer.zero_grad(set_to_none=True)

        with amp.autocast():
            pred = get_forecast_prediction(model, timeseries, input_mask)

            if pred.shape != target.shape:
                raise ValueError(
                    f"Shapes incompatíveis no treino: pred={pred.shape}, target={target.shape}"
                )

            loss = criterion(pred, target)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        train_losses.append(loss.item())

    avg_train_loss = float(np.mean(train_losses))

    # =====================================================
    # Avaliação
    # =====================================================
    model.eval()
    test_losses = []
    trues = []
    preds = []

    with torch.no_grad():
        for timeseries, forecast, input_mask in tqdm(test_loader, desc=f"Test {epoch+1}/{MAX_EPOCHS}"):
            timeseries = timeseries.float().to(device)
            forecast = forecast.float().to(device)
            input_mask = input_mask.to(device)

            timeseries = timeseries.permute(0, 2, 1)  # (B, 17, 128)

            if target_mode == "BCT":
                target = forecast.permute(0, 2, 1)
            else:
                target = forecast

            with amp.autocast():
                pred = get_forecast_prediction(model, timeseries, input_mask)

                if pred.shape != target.shape:
                    raise ValueError(
                        f"Shapes incompatíveis na avaliação: pred={pred.shape}, target={target.shape}"
                    )

                loss = criterion(pred, target)

            test_losses.append(loss.item())
            trues.append(target.detach().cpu().numpy())
            preds.append(pred.detach().cpu().numpy())

    avg_test_loss = float(np.mean(test_losses))

    trues = np.concatenate(trues, axis=0)
    preds = np.concatenate(preds, axis=0)

    mse, mae = compute_metrics(trues, preds)

    print(
        f"Epoch {epoch+1}/{MAX_EPOCHS} | "
        f"Train Loss: {avg_train_loss:.6f} | "
        f"Test Loss: {avg_test_loss:.6f} | "
        f"Test MSE: {mse:.6f} | "
        f"Test MAE: {mae:.6f}"
    )


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

timeseries, forecast, input_mask = next(iter(test_loader))

timeseries = timeseries.float().to(device)
forecast = forecast.float().to(device)
input_mask = input_mask.to(device)

timeseries_model = timeseries.permute(0, 2, 1)  # (B, 17, 128)

model.eval()
with torch.no_grad():
    pred = get_forecast_prediction(model, timeseries_model, input_mask)

timeseries_np = timeseries.detach().cpu().numpy()   # (B, 128, 17)
forecast_np = forecast.detach().cpu().numpy()
pred_np = pred.detach().cpu().numpy()

if pred_np.shape[1] == N_CHANNELS and pred_np.shape[2] == FORECAST_HORIZON:
    pred_np = np.transpose(pred_np, (0, 2, 1))  # (B, 128, 17)

# desnormalização
timeseries_np = timeseries_np * std + mean
forecast_np = forecast_np * std + mean
pred_np = pred_np * std + mean

sample_idx = 0

history_sample = timeseries_np[sample_idx]
target_sample = forecast_np[sample_idx]
pred_sample = pred_np[sample_idx]

n_sensors = history_sample.shape[1]
n_cols = 2
n_rows = int(np.ceil(n_sensors / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 3.5 * n_rows), sharex=False)
axes = axes.flatten()

x_hist = np.arange(history_sample.shape[0])
x_future = np.arange(history_sample.shape[0], history_sample.shape[0] + target_sample.shape[0])

for sensor_idx in range(n_sensors):
    ax = axes[sensor_idx]

    ax.plot(x_hist, history_sample[:, sensor_idx], label="Histórico", color="blue")
    ax.plot(x_future, target_sample[:, sensor_idx], label="Real", color="green")
    ax.plot(x_future, pred_sample[:, sensor_idx], label="Previsto", color="red", linestyle="--")

    ax.set_title(f"Sensor {sensor_idx}")
    ax.set_xlabel("Tempo")
    ax.set_ylabel("Valor")
    ax.grid(True, alpha=0.3)
    ax.legend()

for i in range(n_sensors, len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
plt.show()
